In [0]:
from pyspark.sql.functions import current_timestamp, col

base_volume = "/Volumes/workspace/default/healthflow_raw/raw_data"

batch_files = {
    "patients": "patients.csv",
    "conditions": "conditions.csv",
    "procedures": "procedures.csv",
    "claims": "claims.csv",
    "medications": "medications.csv",
    "careplans": "careplans.csv"
}

for table_name, file_name in batch_files.items():
    file_path = f"{base_volume}/{file_name}"
    
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path)
    )
    
    # ═══════════════════════════════════════════════════════
    # FIX: Strip spaces from column names
    # " Total" → "Total"
    # ═══════════════════════════════════════════════════════
    df = df.toDF(*[c.strip() for c in df.columns])
    
    # Optional: lowercase everything too
    # df = df.toDF(*[c.strip().lower() for c in df.columns])
    
    # Add metadata
    df = (
        df
        .withColumn("ingested_at", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
    )
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.bronze.{table_name}")
    )
    
    print(f"✅ Ingested workspace.bronze.{table_name} ({df.count()} rows)")